# Acemi Kaggle runner — incremental eval doğrulaması

Clones the repository, runs the full correctness suite and a bench smoke, pins the v0.8.2 baseline (`1f48474`), proves fixed-depth search equivalence, then runs a 150-game A/B SPRT against that baseline. The A/B gate is `accept_h1` with `elo0=0`, `elo1=5`; only that result unlocks the 100-game self-play stress and the Stockfish 1500/1320 checkpoints. The book stays off. Pushes only `data/` records. Keep Internet enabled.

The code commit is taken from the latest commit that touched code paths (engine/tools/tests/openings/notebooks) and passed to every match, so `data:` commits cannot be mistaken for the engine commit. Run the cells in order. Approximate durations on a CPU Kaggle session: tests 6–8 min, fixed-depth comparison 2–5 min, A/B 15–40 min, stress 5–10 min, SF1500 30–60 min, SF1320 15–30 min. The test and equivalence cells are required; if the A/B is not `accept_h1`, the runner stops before the validation matches.

In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "main"
WORKSPACE = "/kaggle/working/acemi"


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def run_quietly(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        details = "\n".join(part for part in (result.stdout, result.stderr) if part)
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{details[-4000:]}")
    return result.stdout

repo_url = GITHUB_REPO_URL.strip()
if not repo_url.startswith("https://github.com/") or "YOUR_ACCOUNT" in repo_url:
    raise ValueError("Set GITHUB_REPO_URL to your repository's HTTPS clone URL.")
if "@" in repo_url.split("github.com/", 1)[-1].split("/", 1)[0]:
    raise ValueError("Do not put credentials in GITHUB_REPO_URL. Use the GITHUB_TOKEN Kaggle Secret.")
workspace = Path(WORKSPACE)
if not workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, repo_url, str(workspace)])
else:
    run_quietly(["git", "-C", str(workspace), "pull", "--ff-only", "origin", GITHUB_BRANCH])
os.chdir(workspace)
print(f"Repository ready on branch {GITHUB_BRANCH}.")

print(run_quietly(["git", "-C", str(workspace), "log", "-1", "--format=%h %s"]).strip())
version = subprocess.run(
    ["python", "-c", "import engine; print(engine.__version__)"], text=True, capture_output=True
)
print(f"Engine version: {version.stdout.strip()}")
if version.returncode:
    raise RuntimeError(version.stderr)

# Latest commit that touched code/data files that define the engine, not the data records.
CODE_COMMIT = run_quietly([
    "git", "-C", str(workspace), "log", "-1", "--format=%H", "--",
    "engine", "tools", "tests", "openings", "notebooks", ".github",
]).strip() or run_quietly(["git", "-C", str(workspace), "rev-parse", "HEAD"]).strip()
print(f"Code commit pinned for every run: {CODE_COMMIT[:7]}")

In [ ]:
run_quietly(["apt-get", "update", "-qq"])
run_quietly(["apt-get", "install", "-y", "-qq", "stockfish"])
run_quietly(["python", "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"])
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish package installed but executable was not found.")
version = subprocess.run([stockfish_bin, "--version"], check=True, text=True, capture_output=True).stdout.splitlines()[0]
print(f"Stockfish installed: {version}")
print("Tools installed.")


In [ ]:
# Correctness suite + bench smoke: board, perft, UCI, stats, eval, search, shipped book.
for args in (
    ["python", "-m", "unittest", "discover", "-s", "tests", "-v"],
    ["python", "-m", "tools.bench", "--depth", "3"],
):
    print("$", " ".join(args), flush=True)
    result = subprocess.run(args, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(args)}")
print("Correctness suite and bench smoke passed.")

In [ ]:
# Equivalence gate: the incremental build must search exactly like v0.8.2 at fixed depth.
BASELINE_COMMIT = "1f48474699f9fc0690528aa81b2063966f705b63"
baseline_workspace = Path("/kaggle/working/acemi-baseline")
if not baseline_workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL.strip(), str(baseline_workspace)])
run_quietly(["git", "-C", str(baseline_workspace), "fetch", "origin", BASELINE_COMMIT])
run_quietly(["git", "-C", str(baseline_workspace), "checkout", "--force", BASELINE_COMMIT])
baseline_head = run_quietly(["git", "-C", str(baseline_workspace), "rev-parse", "HEAD"]).strip()
if baseline_head != BASELINE_COMMIT:
    raise RuntimeError(f"Baseline mismatch: {baseline_head} != {BASELINE_COMMIT}")
print(f"Baseline ready: {baseline_head[:7]}")

benchmark_cases = (
    [
        "python", "-m", "tools.bench", "--depth", "4", "--label", "v0.8.2-baseline",
        "--engine-root", str(baseline_workspace),
        "--out", "data/bench/incremental-baseline.json",
    ],
    [
        "python", "-m", "tools.bench", "--depth", "4", "--label", "incremental",
        "--out", "data/bench/incremental-new.json",
    ],
    [
        "python", "-m", "tools.bench", "--compare",
        "data/bench/incremental-baseline.json", "data/bench/incremental-new.json",
    ],
)
for args in benchmark_cases:
    print("$", " ".join(args), flush=True)
    result = subprocess.run(args, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode:
        raise RuntimeError(f"Fixed-depth equivalence failed: {' '.join(args)}")
print("Fixed-depth search equivalence passed.")

In [ ]:
# Push helper: commits the data/ records and pushes them to GitHub. Used after every match
# and in the last cell. Anything committed earlier but not yet pushed is pushed too.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is unavailable; add it under Add-ons â†’ Secrets.") from None
    if not token:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": auth_value,
        "GIT_TERMINAL_PROMPT": "0",
    })

    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=git_env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stderr + result.stdout).replace(token, "[REDACTED]")
            raise RuntimeError(f"Git command failed ({args[0]}): {details[-1500:]}")
        return result

    git("config", "user.name", "Acemi Kaggle Runner")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    tracked = [name for name in ("data/runs", "data/index.csv", "data/positions", "data/bench") if (workspace / name).exists()]
    git("add", "--", *tracked)
    if git("diff", "--cached", "--quiet", check=False).returncode != 0:
        git("commit", "-m", message)
        print(f"Committed: {message}")
    git("fetch", "origin", GITHUB_BRANCH)
    unpushed = int(git("rev-list", "--count", "FETCH_HEAD..HEAD").stdout.strip())
    if unpushed == 0:
        print("Nothing to push: GitHub already has every local record.")
        return
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()) > 0:
        git("rebase", "FETCH_HEAD")  # GitHub moved on since this session started.
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    local_head = git("rev-parse", "HEAD").stdout.strip()
    remote_line = git("ls-remote", "origin", f"refs/heads/{GITHUB_BRANCH}").stdout.split()
    if not remote_line or remote_line[0] != local_head:
        raise RuntimeError("Push finished but GitHub does not show the new commit; run this cell again.")
    print(f"Pushed to GitHub ({unpushed} commit(s)); {GITHUB_BRANCH} is now at {local_head[:7]}.")

In [ ]:
# A/B gate: new code vs the pinned v0.8.2 baseline. Only accept_h1 continues.
import json

ab_args = [
    "python", "-m", "tools.match",
    "--games", "150",
    "--movetime-ms", "500",
    "--openings", "openings/acemi-64.txt",
    "--opponent-cmd", "python -u -m engine",
    "--opponent-cwd", str(baseline_workspace),
    "--sprt",
    "--elo0", "0",
    "--elo1", "5",
    "--alpha", "0.05",
    "--beta", "0.05",
    "--git-commit", CODE_COMMIT,
]
print("$", " ".join(ab_args), flush=True)
ab = subprocess.run(ab_args, text=True, capture_output=True)
if ab.stdout:
    print(ab.stdout, end="")
if ab.stderr:
    print(ab.stderr, end="")
if ab.returncode:
    raise RuntimeError("Incremental A/B match failed.")

recorded_line = next(
    (line.split("Recorded run:", 1)[1].strip() for line in ab.stdout.splitlines() if "Recorded run:" in line),
    None,
)
if not recorded_line:
    raise RuntimeError("A/B completed but the recorded run directory was not reported.")
run_dir = Path(recorded_line)
if not run_dir.is_absolute():
    run_dir = workspace / run_dir
results = json.loads((run_dir / "results.json").read_text(encoding="utf-8"))
decision = results.get("sprt", {}).get("decision")
print(f"A/B SPRT decision: {decision}")

try:
    publish_results("data: incremental eval A/B vs v0.8.2 baseline")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

if decision != "accept_h1":
    raise RuntimeError(
        f"A/B SPRT stopped with {decision}; not running the Stockfish validation cells."
    )
print("A/B accepted: running validation matches.")

In [ ]:
# Robustness stress: 100 self-play games at 100 ms, paired openings, book off.
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "100",
        "--movetime-ms", "100",
        "--openings", "openings/acemi-64.txt",
        "--opponent-cmd", "python -u -m engine",
        "--opponent-cwd", str(workspace),
        "--git-commit", CODE_COMMIT,
    ],
    check=True,
)
try:
    publish_results("data: stress self-play 100 games at 100 ms")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

In [ ]:
# Final checkpoint 1/2: 150 games vs Stockfish 1500 at 500 ms, book off.
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "150",
        "--movetime-ms", "500",
        "--uci-elo", "1500",
        "--openings", "openings/acemi-64.txt",
        "--stockfish-path", stockfish_bin,
        "--git-commit", CODE_COMMIT,
    ],
    check=True,
)
try:
    publish_results("data: final checkpoint vs SF1500 at 500 ms (150 games)")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

In [ ]:
# Final checkpoint 2/2: 150 games vs Stockfish 1320 at 250 ms, book off.
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "150",
        "--movetime-ms", "250",
        "--uci-elo", "1320",
        "--openings", "openings/acemi-64.txt",
        "--stockfish-path", stockfish_bin,
        "--git-commit", CODE_COMMIT,
    ],
    check=True,
)
try:
    publish_results("data: final checkpoint vs SF1320 at 250 ms (150 games)")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

In [ ]:
# Final push: sends anything the per-cell pushes missed. Raises if the push fails.
publish_results("data: record incremental eval validation runs")